In [1]:
import pandas as pd
from collections import defaultdict

In [2]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# show all columns
pd.set_option("display.max_columns", None)

In [3]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [4]:
%%R

require('tidyverse')
require('DescTools')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: DescTools


In [5]:
# Load match data

import glob

# Step 1: Get all CSV files in the folder
csv_files = glob.glob("/Users/hazelgandhi/Desktop/tennis-regression/files/*.csv")

# Step 2: Read and concatenate them
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Optional: Check the shape or preview
print(df.shape)
df.head()
df = df.sort_values('tourney_date')

five_set = ['Us Open','Roland Garros', 'Australian Open', 'Wimbledon']
df['five_set'] = df.tourney_name.isin(five_set)


# Initialize Elo ratings
elo_ratings = defaultdict(lambda: 1500)

# Store Elo snapshot *before* each match
elo_snapshots = []

K = 30

def win_prob(rating_i, rating_j):
    return 1 / (1 + 10 ** ((rating_j - rating_i) / 400))

for _, match in df.iterrows():
    winner = match['winner_name']
    loser = match['loser_name']

    rating_winner = elo_ratings[winner]
    rating_loser = elo_ratings[loser]

    # Record pre-match Elo ratings
    elo_snapshots.append({
        'date': match['tourney_date'],
        'surface': match['surface'],
        'tournament': match['tourney_name'],
        'winner': winner,
        'loser': loser,
        'five_set': match['five_set'],
        'winner_elo_before': rating_winner,
        'loser_elo_before': rating_loser
    })

    # Calculate expected outcomes
    expected_win = win_prob(rating_winner, rating_loser)
    expected_loss = 1 - expected_win

    # Update ratings
    elo_ratings[winner] += K * (1 - expected_win)
    elo_ratings[loser] += K * (0 - expected_loss)

# Create DataFrame
elo_df = pd.DataFrame(elo_snapshots)
elo_df

(8979, 49)


,date,surface,tournament,winner,loser,five_set,winner_elo_before,loser_elo_before
0,20220103,Hard,Adelaide 1,Juan Manuel Cerundolo,Alex Bolt,False,1500.000000,1500.000000
1,20220103,Hard,Adelaide 1,Marin Cilic,Thiago Monteiro,False,1500.000000,1500.000000
2,20220103,Hard,Adelaide 1,Laslo Djere,Corentin Moutet,False,1500.000000,1500.000000
3,20220103,Hard,Adelaide 1,Mikael Ymer,Soon Woo Kwon,False,1500.000000,1500.000000
4,20220103,Hard,Adelaide 1,Thanasi Kokkinakis,Frances Tiafoe,False,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...
8974,20241218,Hard,Next Gen Finals,Luca Van Assche,Juncheng Shang,False,1420.401813,1582.974281
8975,20241218,Hard,Next Gen Finals,Nishesh Basavareddy,Juncheng Shang,False,1500.000000,1561.426509
8976,20241218,Hard,Next Gen Finals,Luca Van Assche,Nishesh Basavareddy,False,1441.949585,1517.624705
8977,20241218,Hard,Next Gen Finals,Learner Tien,Arthur Fils,False,1519.809918,1700.206746


In [6]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Randomly assign winner to be Player A or B
winner_is_A = np.random.randint(0, 2, size=len(elo_df)) == 1

# Create transformed DataFrame
elo_transformed = pd.DataFrame({
    'date': elo_df['date'],
    'tournament': elo_df['tournament'],
    'surface' : elo_df['surface'],
    'five_set': elo_df['five_set'],
    'player_A_name': np.where(winner_is_A, elo_df['winner'], elo_df['loser']),
    'player_B_name': np.where(winner_is_A, elo_df['loser'], elo_df['winner']),
    
    'player_A_elo_before': np.where(winner_is_A, elo_df['winner_elo_before'], elo_df['loser_elo_before']),
    'player_B_elo_before': np.where(winner_is_A, elo_df['loser_elo_before'], elo_df['winner_elo_before']),
    
    'player_A_win': winner_is_A.astype(int)
})
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0
...,...,...,...,...,...,...,...,...,...
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1
8975,20241218,Next Gen Finals,Hard,False,Juncheng Shang,Nishesh Basavareddy,1561.426509,1500.000000,0
8976,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Nishesh Basavareddy,1441.949585,1517.624705,1
8977,20241218,Next Gen Finals,Hard,False,Learner Tien,Arthur Fils,1519.809918,1700.206746,1


In [7]:
elo_transformed['elo_difference'] = elo_transformed['player_A_elo_before'] - elo_transformed ['player_B_elo_before']
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0,0.000000
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1,0.000000
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0,0.000000
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0,0.000000
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1,-162.572468
8975,20241218,Next Gen Finals,Hard,False,Juncheng Shang,Nishesh Basavareddy,1561.426509,1500.000000,0,61.426509
8976,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Nishesh Basavareddy,1441.949585,1517.624705,1,-75.675119
8977,20241218,Next Gen Finals,Hard,False,Learner Tien,Arthur Fils,1519.809918,1700.206746,1,-180.396828


### Calculating overall advantage - surface wise advantage

In [8]:
# Calculate surface-specific win percentage
surface_stats = elo_transformed.groupby(['player_A_name', 'surface'])['player_A_win'].agg(['sum', 'count']).reset_index()
surface_stats.columns = ['player', 'surface', 'wins', 'matches']
surface_stats['win_pct'] = surface_stats['wins'] / surface_stats['matches']

# Calculate overall win percentage
overall_stats = elo_transformed.groupby('player_A_name')['player_A_win'].agg(['sum', 'count']).reset_index()
overall_stats.columns = ['player', 'total_wins', 'total_matches']
overall_stats['overall_win_pct'] = overall_stats['total_wins'] / overall_stats['total_matches']

# Merge surface and overall stats
merged = surface_stats.merge(overall_stats, on='player', how='left')
merged['advantage'] = merged['win_pct'] - merged['overall_win_pct']

# Pivot to wide format
advantage_wide = merged.pivot(index='player', columns='surface', values='advantage').reset_index()
advantage_wide.columns.name = None
advantage_wide = advantage_wide.rename(columns=lambda x: f"{x.lower()}_advantage" if x != 'player' else x)
advantage_wide = advantage_wide.fillna(0)


### merging with elo_transformed

In [9]:
elo_transformed = elo_transformed.loc[:, ~elo_transformed.columns.str.contains('_advantage')]  # drop all advantage cols


In [10]:
# Player A merge
adv_A = advantage_wide.rename(columns=lambda x: 'player_A' if x == 'player' else f"{x}_A")
elo_transformed = elo_transformed.merge(adv_A, left_on='player_A_name', right_on='player_A', how='left')
elo_transformed.drop(columns='player_A', inplace=True)

# Player B merge
adv_B = advantage_wide.rename(columns=lambda x: 'player_B' if x == 'player' else f"{x}_B")
elo_transformed = elo_transformed.merge(adv_B, left_on='player_B_name', right_on='player_B', how='left')
elo_transformed.drop(columns='player_B', inplace=True)


In [11]:
elo_transformed['surface'] = elo_transformed['surface'].astype(str)

In [12]:
def get_surface_adv(row, suffix):
    surface = row['surface'].strip().lower()
    col_name = f"{surface}_advantage_{suffix}"
    return row.get(col_name, 0)

elo_transformed['player_A_surface_advantage'] = elo_transformed.apply(lambda row: get_surface_adv(row, 'A'), axis=1)
elo_transformed['player_B_surface_advantage'] = elo_transformed.apply(lambda row: get_surface_adv(row, 'B'), axis=1)


In [13]:
cols_to_drop = [col for col in elo_transformed.columns if '_advantage_A' in col or '_advantage_B' in col]
elo_transformed.drop(columns=cols_to_drop, inplace=True)


In [15]:
elo_transformed = elo_transformed.dropna()


In [16]:
%%R -i elo_transformed

logistic <- glm(player_A_win ~ elo_difference + elo_difference:five_set + player_A_surface_advantage + player_B_surface_advantage, data=elo_transformed, family=binomial)
print(summary(logistic))
print(PseudoR2(logistic, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set + 
    player_A_surface_advantage + player_B_surface_advantage, 
    family = binomial, data = elo_transformed)

Coefficients:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 -0.0420009  0.0232027  -1.810   0.0703 .  
elo_difference               0.0052704  0.0002273  23.183  < 2e-16 ***
player_A_surface_advantage   4.9828623  0.2522190  19.756  < 2e-16 ***
player_B_surface_advantage  -0.5261848  0.1957804  -2.688   0.0072 ** 
elo_difference:five_setTRUE  0.0029025  0.0005624   5.161 2.45e-07 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 12231  on 8823  degrees of freedom
Residual deviance: 10741  on 8819  degrees of freedom
AIC: 10751

Number of Fisher Scoring iterations: 4

 McFadden 
0.1218295 


In [17]:
%%R -i elo_transformed

df <- elo_transformed %>% mutate(
    predict_proba_R = predict(logistic, type="response"),
    predict_R = ifelse(predict_proba_R > .5, 1,0)
) %>% arrange(elo_difference)

df %>% head()

         date           tournament surface five_set       player_A_name
7419 20240527        Roland Garros    Clay     TRUE     Richard Gasquet
6703 20240304 Indian Wells Masters    Hard    FALSE          Luca Nardi
7384 20240527        Roland Garros    Clay     TRUE Christopher Eubanks
6050 20240115      Australian Open    Hard     TRUE        Dino Prizmic
6734 20240304 Indian Wells Masters    Hard    FALSE    Aleksandar Vukic
4705 20230703            Wimbledon   Grass     TRUE       Jeremy Chardy
      player_B_name player_A_elo_before player_B_elo_before player_A_win
7419  Jannik Sinner            1428.232            1990.804            0
6703 Novak Djokovic            1426.978            1988.387            1
7384  Jannik Sinner            1460.359            1994.272            0
6050 Novak Djokovic            1469.346            1989.502            0
6734 Novak Djokovic            1456.782            1959.527            0
4705 Carlos Alcaraz            1471.108            1973.75

In [18]:
%%R 

# Create confusion matrix
conf_mat <- table(df$predict_R, df$player_A_win)
print(conf_mat) 

# Extract TP, FP, FN
TP <- conf_mat[2,2]
FP <- conf_mat[2,1]
FN <- conf_mat[1,2]
# Calculate Precision and Recall
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
# Print
cat("Precision:", precision, "\n")
cat("Recall:", recall, "\n")

   
       0    1
  0 2931 1495
  1 1538 2860
Precision: 0.6502956 
Recall: 0.6567164 
